In [ ]:
# !pip install -q supervision "ultralytics<=8.3.40"
# !pip install pytorch_lightning
# !pip install -U ipykernel

In [9]:
import cv2
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import supervision as sv

from tqdm import tqdm
from ultralytics import YOLO
from supervision.assets import VideoAssets, download_assets
from collections import defaultdict, deque

# from process_video import process_video

import torch
from torch.nn.utils.rnn import pad_sequence
from torch import nn
import pytorch_lightning as pl
import torchmetrics
from torchmetrics import Metric
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import TQDMProgressBar, RichProgressBar
from torch.nn.utils.rnn import pack_padded_sequence
torch.manual_seed(47)

In [10]:
import json
import pickle
import numpy as np

In [11]:
DATASET_PATH = '/mnt/DGX0Raid/databases/vsalcedop_dataset/2016-ITS-BrnoCompSpeed/dataset/'
RESULTS_PATH = '/mnt/DGX0Raid/databases/vsalcedop_dataset/2016-ITS-BrnoCompSpeed/results/'

SESSION = 'session1_center/'

# === CONFIG ===
VIDEO_PATH = DATASET_PATH + SESSION + 'video.avi'
MODEL_PATH = '../models/yolo11s.pt'
CSV_OUT = 'output_data.csv'
VIDEO_OUT = 'output_video.avi'
MASK_PATH = DATASET_PATH + SESSION + "video_mask.png"
GT_PATH = DATASET_PATH + SESSION + "gt_data.pkl"

In [12]:
MEDIAN_RESULTS_PATH = RESULTS_PATH + "resultsCache_median.pkl"
SYSTEM = 'dubska_bmvc14'
SESSION = 'session1'
CAMERA = 'center'

results_file = open(MEDIAN_RESULTS_PATH, 'rb')
results_median = pickle.load(results_file, encoding='latin1')
results_file.close()

matches = results_median[SYSTEM][(SESSION, CAMERA)]['matches']
filtered_matches = list(filter(lambda x: x['valid'] and x['matched'], matches))
map_matches = {match['matchedId']: match for match in filtered_matches} 
data_cars = results_median[SYSTEM][(SESSION, CAMERA)]['data']['cars']
map_cars = {car['id']: car for car in data_cars}


In [13]:
data = []
for match in filtered_matches:
    if match['matchedId'] in map_cars:
        car = map_cars[match['matchedId']]
        if len(car['frames']) >= 50:
            car_data = car
            car_data['gtSpeed'] = match['gtSpeed']
            car_data['coords'] = [(x, y) for x, y in zip(car['posX'], car['posY'])]
            data.append(car_data)

In [14]:
# --- 2. Dataset
class TrackingDataset(Dataset):
    def __init__(self, data):
        self.samples = []
        for d in data:
            coords = d['coords']
            speed = d['gtSpeed']
            if coords and speed is not None:
                self.samples.append((
                    torch.tensor(coords, dtype=torch.float32), 
                    torch.tensor(speed, dtype=torch.float32),
                    len(coords)))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]


In [15]:
# collate_fn con padding y longitudes
def collate_fn_padding(batch):
    batch.sort(key=lambda x: x[2], reverse=True)
    sequences, speeds, lengths = zip(*batch)
    padded_sequences = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_sequences, torch.stack(speeds), torch.tensor(lengths)

In [ ]:
dataset = TrackingDataset(data)
print(f"Tamaño del dataset con {len(dataset)}")

Tamaño del dataset con 396


In [17]:
# --- 3. DataModule para PyTorch Lightning ---
class TrackingDataModule(pl.LightningDataModule):
    def __init__(self, csv_path, seq_length=10, batch_size=16, split_ratio=(0.7, 0.15, 0.15), num_workers=4):
        super().__init__()
        self.csv_path = csv_path
        self.seq_length = seq_length
        self.batch_size = batch_size
        self.split_ratio = split_ratio
        self.num_workers = num_workers

    def setup(self, stage=None):
        # Crear el dataset completo
        dataset = TrackingDataset(self.csv_path, seq_length=self.seq_length)
        total = len(dataset)
        train_size = int(total * self.split_ratio[0])
        val_size = int(total * self.split_ratio[1])
        test_size = total - train_size - val_size
        self.train_dataset, self.val_dataset, self.test_dataset = random_split(dataset, [train_size, val_size, test_size])

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, num_workers=self.num_workers)
    
    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=self.num_workers)

In [ ]:
# Métrica personalizada para regresión: Porcentaje de predicciones con error absoluto menor a un umbral.
class RelativeRegressionAccuracy(Metric):
    def __init__(self, dist_sync_on_step: bool = False):
        super().__init__(dist_sync_on_step=dist_sync_on_step)
        self.add_state("acc_sum", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("total", default=torch.tensor(0.0), dist_reduce_fx="sum")
    
    def update(self, preds: torch.Tensor, target: torch.Tensor):
        preds = preds.view(-1)
        target = target.view(-1)
        # Calcular el score para cada muestra.
        for p, t in zip(preds, target):
            # Manejo de t == 0: si el valor real es 0, consideramos perfecta la predicción si también es 0.
            if t == 0:
                score = 1.0 if p == 0 else 0.0
            else:
                score = 1 - abs(p - t) / abs(t)
                # Asegurarse de que el score no sea negativo.
                score = max(score, 0.0)
            self.acc_sum += score
            self.total += 1

    def compute(self):
        return self.acc_sum / self.total


class TrackingLSTM(pl.LightningModule):
    def __init__(self, input_size=2, hidden_size=50, num_layers=2, output_size=1, lr=0.0001, threshold=5.0, dropout=0.3):
        """
        threshold: margen de error en km/h para considerar una predicción como correcta.
        """
        super().__init__()
        self.save_hyperparameters()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)
        self.loss_fn = nn.MSELoss()
        self.lr = lr
        self.threshold = threshold

        # Métricas para el conjunto de entrenamiento
        self.train_mse = torchmetrics.MeanSquaredError()
        self.train_mae = torchmetrics.MeanAbsoluteError()
        self.train_reg_acc = RelativeRegressionAccuracy()
        
        # Métricas para el conjunto de validación
        self.val_mse = torchmetrics.MeanSquaredError()
        self.val_mae = torchmetrics.MeanAbsoluteError()
        self.val_reg_acc = RelativeRegressionAccuracy()
        
        # Métricas para el conjunto de test
        self.test_mse = torchmetrics.MeanSquaredError()
        self.test_mae = torchmetrics.MeanAbsoluteError()
        self.test_reg_acc = RelativeRegressionAccuracy()


    def forward(self, x, l):
        # x shape: [batch, seq_length, input_size]
        packed = pack_padded_sequence(x, l.cpu(), batch_first=True, enforce_sorted=True)
        lstm_out, _ = self.lstm(packed)
        # Se toma la salida del último timestep
        out = self.dropout(lstm_out[:, -1, :])
        return self.fc(out)
    
    def training_step(self, batch, batch_idx):
        x, y, l = batch
        y_hat = self(x, l)
        loss = self.loss_fn(y_hat.squeeze(), y)     
        self.log("train_loss", loss, prog_bar=True)

        self.train_mse(y_hat.squeeze(), y)
        self.train_rmse = torch.sqrt(self.train_mse.compute())
        self.train_mae(y_hat.squeeze(), y)
        self.train_reg_acc(y_hat.squeeze(), y)

        self.log("train_mse", self.train_mse, on_step=False, on_epoch=True, prog_bar=False, logger=True)
        self.log("train_rmse", self.train_rmse, on_step=False, on_epoch=True, prog_bar=False, logger=True)
        self.log("train_mae", self.train_mae, on_step=False, on_epoch=True, prog_bar=False, logger=True)
        self.log("train_reg_acc", self.train_reg_acc, on_step=False, on_epoch=True, prog_bar=False, logger=True)

        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y, l = batch
        y_hat = self(x, l)
        loss = self.loss_fn(y_hat.squeeze(), y)
        self.log("val_loss", loss, prog_bar=True)

        if y_hat.squeeze().shape == y.shape:
            self.val_mse(y_hat.squeeze(), y)
            self.val_rmse = torch.sqrt(self.val_mse.compute())
            self.val_mae(y_hat.squeeze(), y)
            self.val_reg_acc(y_hat.squeeze(), y)

            self.log("val_mse", self.val_mse, on_step=False, on_epoch=True, prog_bar=False, logger=True)
            self.log("val_rmse", self.val_rmse, on_step=False, on_epoch=True, prog_bar=False, logger=True)
            self.log("val_mae", self.val_mae, on_step=False, on_epoch=True, prog_bar=False, logger=True)
            self.log("val_reg_acc", self.val_reg_acc, on_epoch=True, prog_bar=False, logger=True)

        return loss
    
    def test_step(self, batch, batch_idx):
        x, y, l = batch
        y_hat = self(x, l)
        loss = self.loss_fn(y_hat.squeeze(), y)
        self.log("test_loss", loss)

        self.test_mse(y_hat.squeeze(), y)
        self.test_rmse = torch.sqrt(self.test_mse.compute())
        self.test_mae(y_hat.squeeze(), y)
        self.test_reg_acc(y_hat.squeeze(), y)

        self.log("test_mse", self.test_mse, on_epoch=True)
        self.log("test_rmse", self.test_rmse, on_epoch=True)
        self.log("test_mae", self.test_mae, on_epoch=True)
        self.log("test_reg_acc", self.test_reg_acc, on_epoch=True)

        return loss
    
    def predict_step(self, batch, batch_idx, dataloader_idx=None):
        x, y, l = batch
        y_hat = self(x, l)
        return y_hat.squeeze(), y
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=1e-5)

dataset = TrackingDataset(data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn_padding, num_workers=4)
model = TrackingLSTM(
    input_size=2,
    hidden_size=64,
    num_layers=2,
    output_size=1,
    lr=0.0001,
    dropout=0
)

trainer = pl.Trainer(max_epochs=20)
trainer.fit(model, dataloader)

In [49]:
seq_length = 20
batch_size = 4
split_ratio = (0.8, 0.1, 0.1)
n_epochs = 200
num_workers = 4

data_module = TrackingDataModule(
    csv_path=OUTPUT_CSV, 
    seq_length=seq_length, 
    batch_size=batch_size, 
    split_ratio=split_ratio,
    num_workers=num_workers
)

model = TrackingLSTM(
    input_size=4,
    hidden_size=128,
    num_layers=2,
    output_size=1,
    lr=0.0001,
    dropout=0.3
)

In [ ]:
logger = CSVLogger("logs/", name="tracking_lstm")

callback_checkpoint = ModelCheckpoint(
    save_top_k=1,
    mode='min', 
    monitor='val_loss'
)
callback_tqdm = TQDMProgressBar(leave=True)

early_stopping_callback = EarlyStopping(monitor='val_loss', patience=7)

trainer = pl.Trainer(
    max_epochs=n_epochs,
    callbacks=[callback_checkpoint, early_stopping_callback, callback_tqdm],
    accelerator="auto",
    devices="auto",
    logger=logger,
    deterministic=False,
    log_every_n_steps=10,
)
trainer.fit(model, datamodule=data_module)

In [ ]:
predictions = trainer.test(model, datamodule=data_module)

In [ ]:
predictions = trainer.predict(model, dataloaders=data_module.test_dataloader())
pred_df = pd.DataFrame(predictions, columns=["pred", "target"])

In [ ]:
df_pred = pd.DataFrame(
    [
        (
            pred[i].item(),
            real[i].item(),
            abs(pred[i].item() - real[i].item())
        )
        for (pred, real) in predictions
        for i in range(len(pred))
    ],
    columns=["Velocidad Predicha", "Velocidad Real", "Error km/h"]
)
df_pred.head(20)

In [ ]:
metrics = pd.read_csv(f"{trainer.logger.log_dir}/metrics.csv")

aggreg_metrics = []
agg_col = "epoch"
for i, dfg in metrics.groupby(agg_col):
    agg = dict(dfg.mean())
    agg[agg_col] = i
    aggreg_metrics.append(agg)

df_metrics = pd.DataFrame(aggreg_metrics)

df_metrics[["train_loss", "val_loss"]].plot(
    grid=True, 
    legend=True, 
    xlabel="Epoch",
    ylabel="MSE Loss", 
    # ylim=(0, 500),
    title="MSE de Entrenamiento y Validación", 
)

df_metrics[["train_rmse", "val_rmse"]].plot(
    grid=True, 
    legend=True, 
    xlabel="Epoch", 
    ylabel="RMSE",
    title="RMSE de Entrenamiento y Validación",
)

df_metrics[["train_mae", "val_mae"]].plot(
    grid=True, 
    legend=True, 
    xlabel="Epoch", 
    ylabel="MAE",
    title="MAE - Entrenamiento y Validación",
)

df_metrics[["train_reg_acc", "val_reg_acc"]].plot(
    grid=True, 
    legend=True, 
    xlabel="Epoch", 
    ylabel="Accuracy",
    title="Precisión Relativa de Entrenamiento y Validación",
)

plt.show()

In [ ]:
print(model)
for nombre, parametro in model.named_parameters():
    print(nombre, parametro.shape)

In [ ]:
best_ckpt = trainer.checkpoint_callback.best_model_path
lstm_model = TrackingLSTM.load_from_checkpoint(best_ckpt)
lstm_model.eval()